# Notebook 02: Pipeline, Validación Cruzada y GridSearchCV

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

import matplotlib.pyplot as plt

Carga y preprocesamiento de datos

In [3]:

# 1. Cargar datos
df = pd.read_csv("heart.csv")

display(df.head())
print(df.info())
print(df.isnull().sum())


# 2. Separar X e y
target_col = "HeartDisease"

X = df.drop(target_col, axis=1)
y = df[target_col]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

print("Columnas numéricas:")
print(list(numeric_features))

print("\nColumnas categóricas:")
print(list(categorical_features))


# 3. División antes del preprocesamiento
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# 4. Preprocesamiento dentro del Pipeline
preprocessor_minmax = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

preprocessor_standard = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB
None
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak          

Función para entrenar modelos

In [4]:
results = []
trained_models = {}

def train_evaluate_model(model_name, pipeline, param_grid):
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    y_pred = grid.predict(X_test)
    y_proba = grid.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_proba)
    acc = accuracy_score(y_test, y_pred)
    
    results.append({
        "Modelo": model_name,
        "AUC": auc,
        "Accuracy": acc,
        "Mejores hiperparámetros": grid.best_params_
    })
    
    trained_models[model_name] = grid
    
    print(f"Modelo: {model_name}")
    print(f"AUC: {auc:.3f}")
    print(f"Accuracy: {acc:.3f}")
    print("Mejores hiperparámetros:")
    print(grid.best_params_)
    
    return grid

SVR

In [5]:
pipe_svc = Pipeline([
    ("preprocessor", preprocessor_minmax),
    ("clf", SVC(probability=True, random_state=42))
])

param_grid_svc = {
    "clf__C": [0.1, 1, 10],
    "clf__gamma": [0.01, 0.1, 1],
    "clf__kernel": ["rbf"]
}

grid_svc = train_evaluate_model(
    model_name="SVC",
    pipeline=pipe_svc,
    param_grid=param_grid_svc
)

Modelo: SVC
AUC: 0.928
Accuracy: 0.891
Mejores hiperparámetros:
{'clf__C': 1, 'clf__gamma': 0.1, 'clf__kernel': 'rbf'}


Regresión Logística

In [6]:
pipe_logistic = Pipeline([
    ("preprocessor", preprocessor_standard),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

param_grid_logistic = {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__penalty": ["l2"],
    "clf__solver": ["liblinear"]
}

grid_logistic = train_evaluate_model(
    model_name="LogisticRegression",
    pipeline=pipe_logistic,
    param_grid=param_grid_logistic
)

Modelo: LogisticRegression
AUC: 0.932
Accuracy: 0.891
Mejores hiperparámetros:
{'clf__C': 0.1, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}


Regresión Logística Ridge con penalización en L2

In [7]:
pipe_ridge = Pipeline([
    ("preprocessor", preprocessor_standard),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

param_grid_ridge = {
    "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "clf__penalty": ["l2"],
    "clf__solver": ["liblinear"]
}

grid_ridge = train_evaluate_model(
    model_name="Ridge Logistic Regression",
    pipeline=pipe_ridge,
    param_grid=param_grid_ridge
)

Modelo: Ridge Logistic Regression
AUC: 0.932
Accuracy: 0.891
Mejores hiperparámetros:
{'clf__C': 0.1, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}


Regresión Logística Lasso con penalización en L1

In [8]:
pipe_lasso = Pipeline([
    ("preprocessor", preprocessor_standard),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

param_grid_lasso = {
    "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "clf__penalty": ["l1"],
    "clf__solver": ["liblinear"]
}

grid_lasso = train_evaluate_model(
    model_name="Lasso Logistic Regression",
    pipeline=pipe_lasso,
    param_grid=param_grid_lasso
)

Modelo: Lasso Logistic Regression
AUC: 0.923
Accuracy: 0.886
Mejores hiperparámetros:
{'clf__C': 0.1, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}


KNN

In [9]:
pipe_knn = Pipeline([
    ("preprocessor", preprocessor_minmax),
    ("clf", KNeighborsClassifier())
])

param_grid_knn = {
    "clf__n_neighbors": [3, 5, 7, 9, 11],
    "clf__weights": ["uniform", "distance"],
    "clf__metric": ["euclidean", "manhattan"]
}

grid_knn = train_evaluate_model(
    model_name="KNN",
    pipeline=pipe_knn,
    param_grid=param_grid_knn
)

Modelo: KNN
AUC: 0.931
Accuracy: 0.875
Mejores hiperparámetros:
{'clf__metric': 'manhattan', 'clf__n_neighbors': 11, 'clf__weights': 'uniform'}


Random Forest

In [10]:
pipe_rf = Pipeline([
    ("preprocessor", preprocessor_minmax),
    ("clf", RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [None, 3, 5, 10],
    "clf__max_features": ["sqrt", "log2"],
    "clf__min_samples_split": [2, 5]
}

grid_rf = train_evaluate_model(
    model_name="RandomForest",
    pipeline=pipe_rf,
    param_grid=param_grid_rf
)

Modelo: RandomForest
AUC: 0.936
Accuracy: 0.897
Mejores hiperparámetros:
{'clf__max_depth': 5, 'clf__max_features': 'sqrt', 'clf__min_samples_split': 2, 'clf__n_estimators': 100}


Gradient Boosting

In [11]:
pipe_gb = Pipeline([
    ("preprocessor", preprocessor_minmax),
    ("clf", GradientBoostingClassifier(random_state=42))
])

param_grid_gb = {
    "clf__n_estimators": [100, 200],
    "clf__learning_rate": [0.01, 0.05, 0.1],
    "clf__max_depth": [2, 3, 4]
}

grid_gb = train_evaluate_model(
    model_name="GradientBoosting",
    pipeline=pipe_gb,
    param_grid=param_grid_gb
)

Modelo: GradientBoosting
AUC: 0.931
Accuracy: 0.902
Mejores hiperparámetros:
{'clf__learning_rate': 0.05, 'clf__max_depth': 2, 'clf__n_estimators': 100}


XGBoost

In [12]:
from xgboost import XGBClassifier
pipe_xgb = Pipeline([
    ("preprocessor", preprocessor_minmax),
    ("clf", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])

param_grid_xgb = {
    "clf__n_estimators": [100, 200],
    "clf__learning_rate": [0.01, 0.05, 0.1],
    "clf__max_depth": [2, 3, 4],
    "clf__subsample": [0.8, 1.0],
    "clf__colsample_bytree": [0.8, 1.0]
}

grid_xgb = train_evaluate_model(
    model_name="XGBoost",
    pipeline=pipe_xgb,
    param_grid=param_grid_xgb
)

Modelo: XGBoost
AUC: 0.933
Accuracy: 0.886
Mejores hiperparámetros:
{'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.05, 'clf__max_depth': 4, 'clf__n_estimators': 100, 'clf__subsample': 0.8}


Ranking final

In [13]:
results_df = pd.DataFrame(results)

ranking = results_df.sort_values(
    by=["AUC", "Accuracy"],
    ascending=False
).reset_index(drop=True)

ranking

ranking[["Modelo", "AUC", "Accuracy", "Mejores hiperparámetros"]]

,Modelo,AUC,Accuracy,Mejores hiperparámetros
0,RandomForest,0.935796,0.896739,"{'clf__max_depth': 5, 'clf__max_features': 'sq..."
1,XGBoost,0.933166,0.885870,"{'clf__colsample_bytree': 0.8, 'clf__learning_..."
2,LogisticRegression,0.932090,0.891304,"{'clf__C': 0.1, 'clf__penalty': 'l2', 'clf__so..."
3,Ridge Logistic Regression,0.932090,0.891304,"{'clf__C': 0.1, 'clf__penalty': 'l2', 'clf__so..."
4,KNN,0.931014,0.875000,"{'clf__metric': 'manhattan', 'clf__n_neighbors..."
5,GradientBoosting,0.930954,0.902174,"{'clf__learning_rate': 0.05, 'clf__max_depth':..."
6,SVC,0.927905,0.891304,"{'clf__C': 1, 'clf__gamma': 0.1, 'clf__kernel'..."
7,Lasso Logistic Regression,0.923362,0.885870,"{'clf__C': 0.1, 'clf__penalty': 'l1', 'clf__so..."


In [14]:
import plotly.express as px
import plotly.io as pio
from IPython.display import HTML, display

# Asegurar que ranking esté ordenado
ranking = ranking.sort_values(by="AUC", ascending=True).reset_index(drop=True)

# =========================
# Gráfica AUC
# =========================
auc_min = ranking["AUC"].min()
auc_max = ranking["AUC"].max()
auc_pad = 0.01

fig_auc = px.bar(
    ranking,
    x="AUC",
    y="Modelo",
    orientation="h",
    text="AUC",
    title="Comparación de modelos según AUC"
)

fig_auc.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig_auc.update_layout(
    xaxis_title="AUC",
    yaxis_title="Modelo",
    yaxis=dict(categoryorder="total ascending"),
    xaxis=dict(range=[max(0, auc_min - auc_pad), min(1, auc_max + auc_pad)]),
    height=500
)

display(HTML(
    pio.to_html(
        fig_auc,
        full_html=False,
        include_plotlyjs="cdn"
    )
))

# =========================
# Gráfica Accuracy
# =========================
ranking_acc = ranking.sort_values(by="Accuracy", ascending=True).reset_index(drop=True)

acc_min = ranking_acc["Accuracy"].min()
acc_max = ranking_acc["Accuracy"].max()
acc_pad = 0.01

fig_acc = px.bar(
    ranking_acc,
    x="Accuracy",
    y="Modelo",
    orientation="h",
    text="Accuracy",
    title="Comparación de modelos según Accuracy"
)

fig_acc.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig_acc.update_layout(
    xaxis_title="Accuracy",
    yaxis_title="Modelo",
    yaxis=dict(categoryorder="total ascending"),
    xaxis=dict(range=[max(0, acc_min - acc_pad), min(1, acc_max + acc_pad)]),
    height=500
)

display(HTML(
    pio.to_html(
        fig_acc,
        full_html=False,
        include_plotlyjs=False
    )
))

Selección y evaluación del mejor modelo

In [15]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    accuracy_score
)

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

# =========================
# Mejor modelo
# =========================
best_model_name = ranking.iloc[0]["Modelo"]
best_grid = trained_models[best_model_name]
best_model = best_grid.best_estimator_

print("Mejor modelo:", best_model_name)
print("Mejores hiperparámetros:")
print(best_grid.best_params_)

# Predicciones
y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

# Reporte de clasificación
print(classification_report(y_test, y_pred_best))

# =========================
# Matriz de confusión interactiva
# =========================
cm = confusion_matrix(y_test, y_pred_best)

fig_cm = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale="Blues",
    x=["Predicción: 0", "Predicción: 1"],
    y=["Real: 0", "Real: 1"],
    title=f"Matriz de confusión - {best_model_name}",
    aspect="auto"
)

fig_cm.update_layout(
    xaxis_title="Clase predicha",
    yaxis_title="Clase real",
    height=500
)

display(HTML(
    pio.to_html(
        fig_cm,
        full_html=False,
        include_plotlyjs="cdn"
    )
))

# =========================
# Curva ROC interactiva
# =========================
fpr, tpr, _ = roc_curve(y_test, y_proba_best)
auc_final = roc_auc_score(y_test, y_proba_best)

fig_roc = go.Figure()

fig_roc.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"ROC (AUC = {auc_final:.3f})"
    )
)

fig_roc.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Línea base",
        line=dict(dash="dash")
    )
)

fig_roc.update_layout(
    title=f"Curva ROC - {best_model_name}",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=500
)

display(HTML(
    pio.to_html(
        fig_roc,
        full_html=False,
        include_plotlyjs=False
    )
))

# =========================
# Métricas finales
# =========================
print(f"AUC final: {auc_final:.3f}")
print(f"Accuracy final: {accuracy_score(y_test, y_pred_best):.3f}")

Mejor modelo: Lasso Logistic Regression
Mejores hiperparámetros:
{'clf__C': 0.1, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}
              precision    recall  f1-score   support

           0       0.90      0.84      0.87        82
           1       0.88      0.92      0.90       102

    accuracy                           0.89       184
   macro avg       0.89      0.88      0.88       184
weighted avg       0.89      0.89      0.89       184



AUC final: 0.923
Accuracy final: 0.886


In [16]:
import joblib
import os

os.makedirs("../app", exist_ok=True)

joblib.dump(best_model, "../app/model.joblib")

print("Modelo guardado en ../app/model.joblib")

Modelo guardado en ../app/model.joblib


In [17]:
# =========================
# Monitoreo de deriva con Evidently
# =========================

import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

# Datos de referencia: entrenamiento
reference_data = X_train.copy()

# Datos actuales: prueba
current_data = X_test.copy()

# Asegurar formato DataFrame
reference_data = pd.DataFrame(reference_data)
current_data = pd.DataFrame(current_data)

# Crear reporte de deriva
drift_report = Report(metrics=[
    DataDriftPreset()
])

# Ejecutar comparación
drift_report.run(
    reference_data=reference_data,
    current_data=current_data
)

# Guardar reporte en la raíz del proyecto
drift_report.save_html("../drift_report.html")

print("Reporte de deriva guardado en ../drift_report.html")

Reporte de deriva guardado en ../drift_report.html
